In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(
    "../data/raw/istanbul_apartment_prices_2026.csv"
)

df_clean = df.copy()

confirmed_error_mask = (
    (df_clean["price"] > 500_000_000)
    |
    (df_clean["net_sqm"] > df_clean["gross_sqm"])
)

df_clean = df_clean.loc[
    ~confirmed_error_mask
].copy()

print(df_clean.shape)

(24753, 30)


In [3]:
df_clean["usable_area_ratio"] = (
    df_clean["net_sqm"]
    /
    df_clean["gross_sqm"]
)

In [4]:
df_clean[
    [
        "gross_sqm",
        "net_sqm",
        "usable_area_ratio"
    ]
].head(10)

,gross_sqm,net_sqm,usable_area_ratio
0,138.0,96.0,0.695652
1,199.0,150.0,0.753769
2,107.0,81.0,0.757009
3,200.0,196.0,0.980000
4,160.0,145.0,0.906250
5,120.0,100.0,0.833333
6,160.0,150.0,0.937500
7,59.0,48.0,0.813559
8,190.0,160.0,0.842105
9,160.0,135.0,0.843750


In [5]:
df_clean["sqm_difference"] = (
    df_clean["gross_sqm"]
    -
    df_clean["net_sqm"]
)

In [6]:
df_clean[
    [
        "gross_sqm",
        "net_sqm",
        "sqm_difference"
    ]
].head()

,gross_sqm,net_sqm,sqm_difference
0,138.0,96.0,42.0
1,199.0,150.0,49.0
2,107.0,81.0,26.0
3,200.0,196.0,4.0
4,160.0,145.0,15.0


In [7]:
df_clean["bathroom_per_room"] = (
    df_clean["bathroom_count"]
    /
    df_clean["total_rooms"]
)

In [8]:
df_clean["floor_ratio"] = (
    df_clean["floor"]
    /
    df_clean["total_floors"]
)

In [9]:
df_clean["is_new_building"] = (
    df_clean["building_age"]
    <=5
).astype(int)

In [10]:
bins = [
    -1,
    5,
    10,
    20,
    40,
    100
]

labels = [
    "0-5",
    "6-10",
    "11-20",
    "21-40",
    "40+"
]

df_clean["building_age_group"] = pd.cut(
    df_clean["building_age"],
    bins=bins,
    labels=labels
)

In [11]:
df_clean[
    [
        "building_age",
        "building_age_group"
    ]
].head(20)

,building_age,building_age_group
0,55,40+
1,0,0-5
2,54,40+
3,35,21-40
4,40,21-40
5,35,21-40
6,30,21-40
7,50,40+
8,25,21-40
9,35,21-40


In [12]:
df_clean["is_large_house"] = (
    df_clean["net_sqm"]
    >=150
).astype(int)

In [13]:
df_clean["has_multiple_bathrooms"] = (
    df_clean["bathroom_count"]
    >=2
).astype(int)

In [14]:
df_clean["sqm_per_room"] = (
    df_clean["net_sqm"]
    /
    df_clean["total_rooms"]
)

In [15]:
print(df_clean.shape)

(24753, 39)


In [16]:
new_features = [
    "usable_area_ratio",
    "sqm_difference",
    "bathroom_per_room",
    "floor_ratio",
    "is_new_building",
    "building_age_group",
    "is_large_house",
    "has_multiple_bathrooms",
    "sqm_per_room"
]

df_clean[
    new_features
].describe(include="all")

,usable_area_ratio,sqm_difference,bathroom_per_room,floor_ratio,is_new_building,building_age_group,is_large_house,has_multiple_bathrooms,sqm_per_room
count,24753.000000,24753.000000,24518.000000,22837.000000,24753.000000,24739,24753.000000,24753.000000,24753.000000
unique,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,0-5,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,10760,NaN,NaN,NaN
mean,0.852908,18.949259,0.417516,0.430012,0.434695,NaN,0.156991,0.411546,29.069351
std,0.080150,17.460340,0.172536,0.413689,0.495727,NaN,0.363800,0.492124,7.095843
min,0.085714,0.000000,0.008547,-1.000000,0.000000,NaN,0.000000,0.000000,0.083333
25%,0.812500,10.000000,0.333333,0.100000,0.000000,NaN,0.000000,0.000000,25.000000
50%,0.869565,15.000000,0.333333,0.428571,0.000000,NaN,0.000000,0.000000,28.000000
75%,0.909091,22.000000,0.500000,0.785714,1.000000,NaN,0.000000,1.000000,32.000000


In [17]:
invalid_floor_ratio = (
    (df_clean["floor_ratio"] < -1)
    |
    (df_clean["floor_ratio"] > 1.5)
)

df_clean.loc[
    invalid_floor_ratio,
    "floor_ratio"
] = np.nan

In [18]:
df_clean["floor_ratio"].describe()

count    22820.000000
mean         0.428448
std          0.409234
min         -1.000000
25%          0.100000
50%          0.428571
75%          0.785714
max          1.500000
Name: floor_ratio, dtype: float64

In [19]:
df_clean.nlargest(
    10,
    "sqm_per_room"
)[
    [
        "price",
        "net_sqm",
        "total_rooms",
        "sqm_per_room",
        "district"
    ]
]

,price,net_sqm,total_rooms,sqm_per_room,district
20479,6000000,250.0,1,250.0,Şişli
20380,49500000,450.0,3,150.0,Şişli
15561,65000000,720.0,5,144.0,Küçükçekmece
7825,25000000,750.0,6,125.0,Büyükçekmece
10430,6475000,120.0,1,120.0,Eyüpsultan
5826,38000000,99.0,1,99.0,Beşiktaş
6210,445000000,390.0,4,97.5,Beykoz
11858,9600000,97.0,1,97.0,Gaziosmanpaşa
3626,82000000,380.0,4,95.0,Bakırköy
6018,7400000,95.0,1,95.0,Beşiktaş


In [20]:
import os

os.makedirs("../data/processed", exist_ok=True)

df_clean.to_csv(
    "../data/processed/featured_data.csv",
    index=False
)

print("featured_data.csv kaydedildi.")

featured_data.csv kaydedildi.
